# Evaluation

Compare base vs DPO-aligned model behavior on held-out prompts.


In [ ]:
import os
import json
import torch
import asyncio
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm
from asyncio import Semaphore
from pydantic_settings import BaseSettings, SettingsConfigDict

from evaluation_helpers import (
    extract_qa,
    run_local_inference,
    judge_with_openai,
)

In [ ]:
from pathlib import Path
from pydantic_settings import BaseSettings, SettingsConfigDict
from openai import AsyncOpenAI
from asyncio import Semaphore

# Resolve repo root
REPO_ROOT = Path(__file__).resolve().parents[1]

class Config(BaseSettings):
    OPENAI_API_KEY: str

    model_config = SettingsConfigDict(
        env_file=REPO_ROOT / ".env",
        env_file_encoding="utf-8",
    )

config = Config()
client = AsyncOpenAI(api_key=config.OPENAI_API_KEY)

semaphore = Semaphore(5)


In [ ]:
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2-7B-Instruct"
JUDGE_MODEL = "gpt-4o-mini"

MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
SAMPLE_LIMIT = 50
EPSILON = 1e-3

REPO_ROOT = Path(__file__).resolve().parents[1]


DPO_MODEL_PATH = REPO_ROOT / "models" / "SkyDPO_Qwen27"
DATASET_PATH   = REPO_ROOT / "Skydpo_dataset"

OUTPUT_JSON = REPO_ROOT / "llm_judge_results.json"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).eval()

dpo_model = AutoModelForCausalLM.from_pretrained(
    DPO_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).eval()

In [ ]:
dataset = load_from_disk(DATASET_PATH)["validation"]
dataset = dataset.select(range(min(SAMPLE_LIMIT, len(dataset))))
print(f"Evaluating {len(dataset)} validation samples")

In [ ]:
async def run_evaluation():
    stats = {
        "base_wins": 0,
        "dpo_wins": 0,
        "ties": 0,
        "base_scores": [],
        "dpo_scores": [],
    }

    records = []
    printed_example = False

    for sample in tqdm(dataset, desc="Evaluating"):
        conv_text = sample["conversations"][0]["value"]
        q, a1, a2 = extract_qa(conv_text)

        if not printed_example:
            print("\nQUESTION:\n", q)
            print("\nANSWER 1:\n", a1)
            print("\nANSWER 2:\n", a2)
            printed_example = True

        prompt = (
            "As an evaluation expert, given a question and its two possible answers, "
            "please select which answer best meets the criteria of coherence, accuracy, "
            "coverage, and overall quality.\n"
            "Output JSON with fields 'reason' and 'better_answer'.\n\n"
            f"Question: {q}\n"
            f"Answer 1: {a1}\n"
            f"Answer 2: {a2}\n"
        )

        base_out = run_local_inference(
            base_model, tokenizer, prompt, MAX_NEW_TOKENS, TEMPERATURE
        )
        dpo_out = run_local_inference(
            dpo_model, tokenizer, prompt, MAX_NEW_TOKENS, TEMPERATURE
        )

        judgment = await judge_with_openai(
            client, semaphore, JUDGE_MODEL,
            q, a1, a2, base_out, dpo_out
        )

        b = float(judgment["base_score"])
        d = float(judgment["dpo_score"])

        stats["base_scores"].append(b)
        stats["dpo_scores"].append(d)

        if abs(d - b) < EPSILON:
            stats["ties"] += 1
            winner = "tie"
        elif d > b:
            stats["dpo_wins"] += 1
            winner = "dpo"
        else:
            stats["base_wins"] += 1
            winner = "base"

        records.append({
            "question": q,
            "base_score": b,
            "dpo_score": d,
            "winner": winner,
            "judge_reason": judgment["reason"],
            "base_output": base_out,
            "dpo_output": dpo_out,
        })

    return stats, records


In [ ]:
stats, records = asyncio.run(run_evaluation())

total = stats["base_wins"] + stats["dpo_wins"]
win_rate = stats["dpo_wins"] / total if total > 0 else 0.0

print("\nRESULTS")
print(f"Base wins : {stats['base_wins']}")
print(f"DPO wins  : {stats['dpo_wins']}")
print(f"Ties      : {stats['ties']}")
print(f"DPO Win Rate : {win_rate:.3f}")
print(f"Avg Base Score : {sum(stats['base_scores'])/len(stats['base_scores']):.2f}")
print(f"Avg DPO Score  : {sum(stats['dpo_scores'])/len(stats['dpo_scores']):.2f}")

with open(OUTPUT_JSON, "w") as f:
    json.dump(records, f, indent=2)

print(f"\nSaved results to {OUTPUT_JSON}")


This completes the **full preference alignment pipeline**:

Raw data → Judge → Preferences → DPO → Evaluation
